# 用 MuJoCo 看一台线性臂

这个 Notebook 会一步步带你：

1. 找到仓库里的机械臂模型文件；
2. 用 MuJoCo 把模型加载进来；
3. 给模型加一个好看的背景；
4. 打开三维窗口，观察机械臂。

**怎么运行**：点一下单元格，按 `Shift + Enter`，从上往下依次执行。
如果中间改过代码，建议点菜单 `Kernel -> Restart & Run All` 从头再跑一遍。

> 需要先安装 `mujoco` 和 `numpy`，并且 Jupyter 要选对 Python 环境。


## 第 1 步：导入工具

Python 用 `import` 把别人写好的功能“拿过来用”。

- `time`：让程序暂停一小会儿，控制画面刷新速度。
- `os`：这里只用它算相对路径。
- `Path`：处理文件路径，比字符串拼接更清楚。
- `mujoco`：物理仿真引擎，负责加载模型、计算运动。
- `numpy`：数值计算库，这里用来处理网格顶点。
- `viewer`：MuJoCo 自带的三维可视化窗口。


In [39]:
import os
import time
from pathlib import Path

import mujoco
import numpy as np
from mujoco import viewer


## 第 2 步：三个开关

下面三个变量就是整个 Notebook 的“设置”。

- `USE_URDF`：选模型格式。`False` 用 MJCF（带闭环约束），`True` 用 URDF。
- `USE_GRAVITY`：是否打开重力。关掉重力，机械臂会安静地停在原地。
- `USE_BACKGROUND`：是否加天空盒和棋盘地面。

改完开关后，从后面的“加载模型”开始重新运行。


In [40]:
USE_URDF = False
USE_GRAVITY = False
USE_BACKGROUND = True


## 第 3 步：找到模型文件

Jupyter 里没有脚本常见的 `__file__`，所以不能直接照搬普通 `.py` 文件的写法。
这里用 `Path.cwd()` 拿到“当前工作目录”，再一层层往上找，直到看见
`model/mjcf/linear_arm.xml` 为止。这样不管从仓库根目录还是 `sim/` 目录启动，
都能找到模型。

`os.path.relpath` 会算出从当前目录到模型文件的相对路径，后面加载时要用它。


In [41]:
def find_repo_root(start):
    """从 start 开始逐级向上找，直到看见 model/mjcf/linear_arm.xml。"""
    for folder in [start] + list(start.parents):
        if (folder / "model" / "mjcf" / "linear_arm.xml").exists():
            return folder
    raise FileNotFoundError(
        "没有找到模型文件。请从仓库目录（或它的子目录）启动 Jupyter，再运行本 Notebook。"
    )


REPO_ROOT = find_repo_root(Path.cwd())
MODEL_DIR = REPO_ROOT / "model"
MJCF_PATH = MODEL_DIR / "mjcf" / "linear_arm.xml"
URDF_PATH = MODEL_DIR / "urdf" / "linear_arm.urdf"

print("仓库目录 :", REPO_ROOT)
print("MJCF 模型:", MJCF_PATH)
print("URDF 模型:", URDF_PATH)


仓库目录 : d:\魏舟\博客博文\linrot_arm\linrot-arm
MJCF 模型: d:\魏舟\博客博文\linrot_arm\linrot-arm\model\mjcf\linear_arm.xml
URDF 模型: d:\魏舟\博客博文\linrot_arm\linrot-arm\model\urdf\linear_arm.urdf


## 第 4 步：加载模型

MuJoCo 把数据分成两份：

- `MjModel`：模型本身，比如质量、关节、执行器，加载后基本不变；
- `MjData`：运行状态，比如每个关节当前的位置、速度，仿真时会不断更新。

`mj_forward` 会根据当前关节位置，把刚体的位置和速度都算一遍。


In [42]:
model_path = URDF_PATH if USE_URDF else MJCF_PATH
if not model_path.exists():
    raise FileNotFoundError(f"找不到模型文件: {model_path}")

# MuJoCo 在部分 Windows 环境里打不开含中文的绝对路径，
# 所以这里换算成相对路径再交给它。
model_load_path = os.path.relpath(model_path, Path.cwd())

print("要加载的模型:", model_path)

model = mujoco.MjModel.from_xml_path(model_load_path)
data = mujoco.MjData(model)

if not USE_GRAVITY:
    model.opt.gravity[:] = [0.0, 0.0, 0.0]

mujoco.mj_forward(model, data)


要加载的模型: d:\魏舟\博客博文\linrot_arm\linrot-arm\model\mjcf\linear_arm.xml


## 第 5 步：看看模型里有什么

`model` 里存着很多信息。下面挑几个常用的打印出来：

- `nbody`：刚体数量；
- `njnt`：关节数量；
- `nq`：位置变量的个数（自由度）；
- `nu`：执行器数量；
- `body_mass`：每个刚体的质量，求和就是总质量。

`f"{...:.4f}"` 表示保留 4 位小数。


In [43]:
print(f"MuJoCo 版本 : {mujoco.__version__}")
print(f"刚体数量    : {model.nbody}")
print(f"关节数量    : {model.njnt}")
print(f"自由度 nq   : {model.nq}")
print(f"执行器数量  : {model.nu}")
print(f"总质量      : {model.body_mass.sum():.4f} kg")
print(f"重力        : {model.opt.gravity}")


MuJoCo 版本 : 3.14.0
刚体数量    : 22
关节数量    : 18
自由度 nq   : 18
执行器数量  : 6
总质量      : 6.3625 kg
重力        : [0. 0. 0.]


## 第 6 步：让机械臂保持当前姿态

这个模型配了 6 个“位置执行器”。位置执行器的意思是：
你告诉它一个目标位置 `ctrl`，它就会把关节往那里推。

刚加载时所有 `ctrl` 都是 0，等于命令机械臂回到零位。
我们想让它在打开窗口时保持现在的姿态，所以把每个执行器的目标
设成对应关节的当前位置。

执行器和关节的对应关系藏在 `model.actuator_trnid` 里：
`model.actuator_trnid[i, 0]` 就是第 `i` 个执行器驱动的关节编号。


In [44]:
def hold_current_pose(model, data):
    """把每个位置执行器的目标设成关节当前值，机械臂就会停在原地。"""
    for i in range(model.nu):
        joint_id = model.actuator_trnid[i, 0]
        qpos_index = model.jnt_qposadr[joint_id]
        data.ctrl[i] = data.qpos[qpos_index]


hold_current_pose(model, data)


## 第 7 步：打印执行器对照表

`mujoco.mj_id2name` 可以把编号翻译成名字。
下面用循环把 6 个执行器以及它们驱动的关节列出来，方便后面做控制实验。


In [45]:
print("编号  执行器      驱动关节")
for i in range(model.nu):
    actuator_name = mujoco.mj_id2name(
        model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    joint_id = model.actuator_trnid[i, 0]
    joint_name = mujoco.mj_id2name(
        model, mujoco.mjtObj.mjOBJ_JOINT, joint_id)
    print(f"  {i}: {actuator_name}  ->  {joint_name}")


编号  执行器      驱动关节
  0: act_R1  ->  R1
  1: act_R2  ->  R2
  2: act_R3  ->  R3
  3: act_L1  ->  L1
  4: act_L2  ->  L2
  5: act_L3  ->  L3


## 第 8 步：加一个好看的背景（进阶，可跳过）

MuJoCo 官方示例里常见的“渐变天空 + 棋盘地面”并不是模型自带的。
下面这段代码会复制一份模型，在复制品上添加背景，再重新编译。

- `MjSpec` 可以理解成“模型图纸”，先改图纸，再 `compile()` 生成新模型；
- 地面要放在机械臂最低点的下面，`mesh_lowest_z` 就是在算这个最低点；
- 背景只影响画面，不参与碰撞，也不影响物理计算。

这段涉及矩阵运算，第一次看不懂没关系，先运行出效果更重要。


In [46]:
def mesh_lowest_z(model, data):
    """算出所有网格最低点的 z 坐标，用来把地面铺在模型脚下。"""
    lowest = np.inf
    for geom_id in range(model.ngeom):
        # 只处理网格类型的几何体
        if model.geom_type[geom_id] != mujoco.mjtGeom.mjGEOM_MESH:
            continue

        mesh_id = model.geom_dataid[geom_id]
        start = model.mesh_vertadr[mesh_id]
        count = model.mesh_vertnum[mesh_id]
        vertices = model.mesh_vert[start:start + count]

        rotation = data.geom_xmat[geom_id].reshape(3, 3)
        world_z = (vertices @ rotation.T + data.geom_xpos[geom_id])[:, 2]
        lowest = min(lowest, float(world_z.min()))

    return float(lowest) if np.isfinite(lowest) else 0.0


In [47]:
def add_classic_background(path, model, data):
    """给模型加天空盒和棋盘地面；成功返回新模型，失败返回 None。"""
    spec = mujoco.MjSpec.from_file(path)

    spec.add_texture(
        name="skybox", type=mujoco.mjtTexture.mjTEXTURE_SKYBOX,
        builtin=mujoco.mjtBuiltin.mjBUILTIN_GRADIENT,
        rgb1=[0.3, 0.5, 0.7], rgb2=[0.0, 0.0, 0.0],
        width=512, height=3072)

    spec.add_texture(
        name="groundplane", type=mujoco.mjtTexture.mjTEXTURE_2D,
        builtin=mujoco.mjtBuiltin.mjBUILTIN_CHECKER,
        mark=mujoco.mjtMark.mjMARK_EDGE,
        rgb1=[0.2, 0.3, 0.4], rgb2=[0.1, 0.2, 0.3],
        markrgb=[0.8, 0.8, 0.8], width=300, height=300)

    spec.add_material(
        name="groundplane", textures=["", "groundplane"],
        texuniform=True, texrepeat=[5, 5], reflectance=0.2)

    floor_z = mesh_lowest_z(model, data) - 0.01
    floor = spec.worldbody.add_geom(
        name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE,
        size=[0, 0, 0.05], pos=[0, 0, floor_z],
        contype=0, conaffinity=0, group=0)
    floor.material = "groundplane"

    new_model = spec.compile()
    material_id = mujoco.mj_name2id(
        new_model, mujoco.mjtObj.mjOBJ_MATERIAL, "groundplane")
    if material_id < 0:
        return None
    return new_model


## 第 9 步：准备用于显示的模型

如果打开了背景开关，就调用上面的函数生成一份带背景的模型；
如果生成失败（比如 URDF 不支持），就继续用原来的模型。

注意：带背景的是一份“新模型”，所以要重新创建 `MjData`，并重新设置重力
和保持姿态。


In [48]:
model_view, data_view = model, data

if USE_BACKGROUND:
    background_model = add_classic_background(
        model_load_path, model, data)
    if background_model is None:
        print("INFO: 这个模型加不了背景，继续用纯色背景")
    else:
        model_view = background_model
        data_view = mujoco.MjData(model_view)
        print("INFO: 已加上天空盒和棋盘地面")

if not USE_GRAVITY:
    model_view.opt.gravity[:] = [0.0, 0.0, 0.0]

mujoco.mj_forward(model_view, data_view)
hold_current_pose(model_view, data_view)


INFO: 已加上天空盒和棋盘地面


## 第 10 步：打开三维窗口

`launch_passive` 会打开一个窗口，但物理推进由我们自己写循环控制：

1. `mj_step` 让仿真往前走一小步；
2. `sync` 把最新状态画到窗口里；
3. `sleep` 稍微等一会儿，避免跑得太快。

这里每次循环推进 10 个物理步（模型步长 1 ms，所以约 10 ms 一帧）。
关闭窗口后，`is_running()` 变成 `False`，循环结束，单元格也就运行完了。

> 窗口开着的时候，这个单元格会一直显示“运行中”，这是正常的。
> 想停止就关闭窗口。


In [49]:
STEPS_PER_FRAME = 10
frame_time = model_view.opt.timestep * STEPS_PER_FRAME

print("正在打开窗口……关闭窗口后本单元格结束。")

with viewer.launch_passive(model_view, data_view) as handle:
    while handle.is_running():
        for _ in range(STEPS_PER_FRAME):
            mujoco.mj_step(model_view, data_view)
        handle.sync()
        time.sleep(frame_time)

print("窗口已关闭。")


正在打开窗口……关闭窗口后本单元格结束。
窗口已关闭。


## 第 11 步：动手改一改

先关闭三维窗口，再运行下面的单元格。

1. 把 `data_view.ctrl[0]` 改成 `0.3`，再重新运行“打开窗口”那一格，
   就能看到 `act_R1` 带着肩膀转起来。
2. 把最上面的 `USE_GRAVITY` 改成 `True`，然后从“加载模型”开始重新运行。
3. 把 `USE_URDF` 改成 `True`，看看和 MJCF 有什么不同
   （URDF 没有闭环约束，也没有执行器）。

`ctrl[0]` 对应哪个关节，可以回看第 7 步打印的对照表。


In [50]:
# 练习 1：让第 1 个执行器（act_R1）转到 0.3 rad
# 取消下面这行的注释，然后重新运行“打开窗口”那一格
# data_view.ctrl[0] = 0.3

# 练习 2：不开窗口，直接在后台仿真 2 秒，看看关节有没有跟上目标
if model_view.nu > 0:
    data_view.ctrl[0] = 0.3
    for _ in range(2000):
        mujoco.mj_step(model_view, data_view)
    print(f"act_R1 目标 0.3 rad，实际 {data_view.qpos[0]:.4f} rad")
else:
    print("当前模型没有执行器，换回 MJCF（USE_URDF=False）再试。")

# 练习 3：打印每个执行器对应的关节当前位置
print("\n各关节当前位置：")
for i in range(model_view.nu):
    joint_id = model_view.actuator_trnid[i, 0]
    qpos_index = model_view.jnt_qposadr[joint_id]
    joint_name = mujoco.mj_id2name(
        model_view, mujoco.mjtObj.mjOBJ_JOINT, joint_id)
    print(f"  {joint_name}: {data_view.qpos[qpos_index]:.4f}")


act_R1 目标 0.3 rad，实际 0.2996 rad

各关节当前位置：
  R1: 0.2996
  R2: 0.0289
  R3: -0.4060
  L1: -0.0088
  L2: 0.0000
  L3: -0.0000


## 小结

这个 Notebook 里最值得记住的几件事：

- `MjModel` 存模型参数，`MjData` 存运行状态；
- `data.qpos` 是关节位置，`data.ctrl` 是执行器目标；
- `mj_step(model, data)` 推进一步仿真；
- `mj_forward(model, data)` 只根据当前位置算一遍，不推进时间；
- 在 Notebook 里没有 `__file__`，找文件要用 `Path.cwd()` 之类的方法。

想继续学控制，可以打开 `sim/control_sim.py`，看位置环、重力前馈和 IK 的写法。

## 遇到问题怎么办

- `ModuleNotFoundError: No module named 'mujoco'`：Jupyter 选错 Python 环境了，
  换成装了 `mujoco` 的那个。
- `FileNotFoundError`：Jupyter 的工作目录不在仓库里，从仓库根目录重新启动。
- 窗口打不开：确认在本地有图形界面的环境运行，不要在没有显示器的服务器上开窗口。
